# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kab-s/flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: Scoring / Ranking**

This is a **scoring and ranking problem**, not binary classification. The goal is to produce a priority score for each page and rank all pages from highest to lowest priority.

Why ranking over classification:
- The editor's constraint is **capacity** (can review ~50 pages/month), not a binary yes/no gate
- A binary classifier would produce ~13,191 "yes" predictions (pages meeting refresh criteria) — still too many to review
- What matters is **which 50 pages to review first** — i.e., the top of a ranked queue
- Ranking allows the editor to work down the list until capacity is exhausted
- The metric from w01 (Precision@50) directly measures ranking quality at the decision boundary

In [22]:
import os, sys, subprocess
import pandas as pd
import numpy as np

# Colab setup
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"

# Load data
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print(f"Loaded {len(df):,} rows × {len(df.columns)} columns")
print(f"Task type: SCORING/RANKING (not binary classification)")

Working dir: d:\flyrank-ml\flyrank
Loaded 30,000 rows × 44 columns
Task type: SCORING/RANKING (not binary classification)


## 2. Target or proxy

**Target: `is_declining_label` (proxy for now, improving in capstone)**

This is a **proxy label** derived from a rule applied to recent time windows, not an observed future outcome. Here's the honest breakdown:

**What the label represents:**
- `is_declining_label = (trend_direction == "down")`
- `trend_direction` is computed from `trend_pct = (impressions_last_30d - impressions_prev_30d) / impressions_prev_30d × 100`
- Logic: "down" if `trend_pct < -20%`
- It's a **current-window comparison** (days 1-30 vs days 31-60), not a forward-looking prediction

**Why it's a proxy (limitation):**
- It tells us "this page showed a declining pattern in the recent past"
- It does NOT tell us "this page will decline in the future"
- The label comes from applying a -20% threshold rule to calculated windows, not an independent measured outcome
- **We're learning patterns that correlate with recent decline, not predicting future decline**

**Why it's adequate for now:**
- The reference pipeline uses this same proxy label and achieves 0.740 P@50
- It's directionally useful: pages showing recent decline ARE more likely to need editorial attention
- **Despite being a proxy, it enables useful ranking** — proven by reference pipeline
- Good enough to establish baseline and test feature improvements

**How I'll improve this in capstone:**
- Use warehouse data to define a **true forward-looking label**
- Structure: Features from months 1-3 → predict decline in month 4
- This creates proper temporal separation (no window overlap)
- Tests: "Can features from the past predict the future?" (real prediction, not correlation)
- **This is a key improvement area to beat the reference pipeline**

In [27]:
# Show how the label is constructed and its distribution

print("LABEL CONSTRUCTION:")
print(f"Label column: is_declining_label")
print(f"Derived from: trend_direction (computed from trend_pct)")
print(f"Rule: is_declining_label = (trend_direction == 'down')")
print()

# Create the label column
df['is_declining_label'] = (df['trend_direction'].str.lower() == 'down')

# Distribution
label_counts = df['is_declining_label'].value_counts()
print("LABEL DISTRIBUTION:")
print(f"Declining (True):  {label_counts.get(True, 0):,} ({label_counts.get(True, 0)/len(df)*100:.1f}%)")
print(f"Not declining (False): {label_counts.get(False, 0):,} ({label_counts.get(False, 0)/len(df)*100:.1f}%)")
print()

print("LABEL TYPE: PROXY (rule-derived, not observed future outcome)")
print("Critical leakage warning: trend_direction and trend_pct CANNOT be features")

LABEL CONSTRUCTION:
Label column: is_declining_label
Derived from: trend_direction (computed from trend_pct)
Rule: is_declining_label = (trend_direction == 'down')

LABEL DISTRIBUTION:
Declining (True):  16,262 (54.2%)
Not declining (False): 13,738 (45.8%)

LABEL TYPE: PROXY (rule-derived, not observed future outcome)
Critical leakage warning: trend_direction and trend_pct CANNOT be features


## 3. Success metric

**Metric: Precision@50**

**Definition:** Of the top 50 pages in the ranked queue, what percentage are actually declining (true positives)?

**Why this metric:**
1. **Matches the editor's constraint:** Editors can review ~50 pages/month, so we care about precision at exactly that cutoff
2. **Interpretable:** "37 out of 50 recommendations were correct" is clear to stakeholders
3. **Actionable:** Editor works down the list; precision@50 directly measures quality at the decision boundary
4. **Comparable across iterations:** Can measure improvement as I iterate on features and models

**How success is measured:**
- Compare against **reference pipeline baseline** on the same data
- Use **client-holdout validation** (entire clients held out, not random rows)
- Improvement = my model's P@50 > baseline P@50 on held-out test set
- Even small improvements matter: +1 correct prediction in top 50 = significant editorial time saved

**Critical validation requirement:**
Must use **CLIENT HOLDOUT** validation (not random row split):
- Entire clients are held out (e.g., 6 clients → test, 26 clients → train)
- Tests: "Can the model generalize to pages from NEW clients it's never seen?"
- This is **harder** than random split but more honest
- Random splits can achieve 0.90+ but are misleading (client patterns leak between train/test)
- Client holdout is how the system will be deployed (new clients join FlyRank)

**My improvement strategy:**
1. **Reproduce baseline** — verify I can match reference pipeline on current data
2. **Feature engineering** — log transforms, interactions, derived ratios
3. **Better models** — XGBoost, hyperparameter tuning, ensembles
4. **Warehouse data** — forward-looking labels, query-level features, time-series signals
5. **Rigorous validation** — always use client holdout to ensure honest comparison

**Why not other metrics:**
- ROC-AUC: doesn't care about the top 50 specifically; high AUC can coexist with poor precision@50
- Recall: we don't need to find ALL declining pages, just the highest-priority 50
- F1: balances precision/recall equally; we care more about precision at the top of the queue

In [24]:
# Simulate precision@50 calculation to show the metric

# Create the label
df['is_declining_label'] = (df['trend_direction'].str.lower() == 'down')

# Simple baseline score for demonstration (visibility × freshness risk)
df['demo_score'] = (
    0.5 * (df['impressions_90d'] / df['impressions_90d'].max()) +
    0.5 * (df['days_since_last_update'] / df['days_since_last_update'].max())
)

# Rank by score and take top 50
top_50 = df.nlargest(50, 'demo_score')
precision_at_50 = top_50['is_declining_label'].sum() / 50

print("PRECISION@50 DEMONSTRATION:")
print(f"Top 50 pages by demo score")
print(f"True positives (actually declining): {top_50['is_declining_label'].sum()}")
print(f"Precision@50: {precision_at_50:.3f} ({top_50['is_declining_label'].sum()}/50)")
print()
print("BENCHMARK TARGETS:")
print(f"  Baseline (hand-rule):  0.240 (12/50)")
print(f"  Acceptable ML:         0.400+ (20/50)")
print(f"  Strong ML:             0.600+ (30/50)")
print(f"  Reference pipeline:    0.740 (37/50)")

PRECISION@50 DEMONSTRATION:
Top 50 pages by demo score
True positives (actually declining): 27
Precision@50: 0.540 (27/50)

BENCHMARK TARGETS:
  Baseline (hand-rule):  0.240 (12/50)
  Acceptable ML:         0.400+ (20/50)
  Strong ML:             0.600+ (30/50)
  Reference pipeline:    0.740 (37/50)


## 4. The unit of analysis, as a real dataframe

**Unit of analysis: One row = one content page**

The dataset grain is one row per pseudonymized content item (page), with trailing-90-day aggregated metrics.

**Key characteristics (starter dataset):**
- **30,000 unique pages** across 32 anonymized clients
- Each row represents a **snapshot** at a point in time (not a time series)
- Metrics are **aggregated over 90 days** (impressions, clicks, sessions, etc.)
- **No filtering applied** — all pages in the dataset are candidates for scoring

**Filters we do NOT apply:**
- We score ALL pages, not just those meeting refresh criteria upfront
- The model decides priority; we don't pre-filter to "declining only" or "high traffic only"
- Filtering would artificially inflate precision by removing hard negatives

**Sample of 10 rows showing the grain:**

In [25]:
# Show the dataset grain and key columns

print("DATASET GRAIN:")
print(f"Total rows: {len(df):,}")
print(f"Unique content_id: {df['content_id'].nunique():,}")
print(f"Unique client_id: {df['client_id'].nunique()}")
print(f"One row = one content page (snapshot with 90-day metrics)")
print()

# Verify grain uniqueness
assert df['content_id'].nunique() == len(df), "content_id is not unique!"
print("✓ Grain verified: content_id is unique (one row per page)")
print()

# Show sample with key columns for scoring
key_cols = [
    'content_id', 'client_id', 'impressions_90d', 'avg_position', 
    'ctr', 'days_since_last_update', 'word_count', 'is_declining_label'
]
print("SAMPLE (10 rows with key scoring signals):")
print(df[key_cols].head(10).to_string(index=False))
print()

print("KEY FACTS:")
print(f"- Pages declining: {df['is_declining_label'].sum():,} ({df['is_declining_label'].sum()/len(df)*100:.1f}%)")
print(f"- Pages stable/up: {(~df['is_declining_label']).sum():,} ({(~df['is_declining_label']).sum()/len(df)*100:.1f}%)")
print(f"- Median impressions: {df['impressions_90d'].median():,.0f}")
print(f"- Median position: {df[df['avg_position'] > 0]['avg_position'].median():.1f}")

DATASET GRAIN:
Total rows: 30,000
Unique content_id: 30,000
Unique client_id: 32
One row = one content page (snapshot with 90-day metrics)

✓ Grain verified: content_id is unique (one row per page)

SAMPLE (10 rows with key scoring signals):
          content_id         client_id  impressions_90d  avg_position  ctr  days_since_last_update  word_count  is_declining_label
content_304f48230142 client_f369cb89fc             3803          10.6 0.76                      20      3221.0                True
content_a1fb4e703a9e client_4e07408562            15320          20.3 0.05                      25      2481.0                True
content_9aa793d4d895 client_7f2253d7e2            12581          36.5 0.09                      20      3515.0                True
content_331d6c4de07b client_19581e27de            11751           6.2 0.49                      22         NaN               False
content_d99b7a2d90ca client_3fdba35f04            19140          44.0 0.13                      14     

## 5. Why ML beats a fixed rule here

**The pattern is real but too tangled for a simple rule.**

A fixed rule (like the baseline: `0.40 × visibility + 0.30 × freshness_risk + 0.25 × position_opportunity + 0.05 × depth_gap`) has to:

1. **Choose fixed weights upfront** — but the importance of freshness vs position vs CTR varies by content type, intent, and traffic level
2. **Use linear combinations** — but interactions matter (e.g., "stale + high-visibility" is more urgent than either signal alone)
3. **Apply one rule to all pages** — but a blog post declining from position 3 is different from a landing page declining from position 15

**Evidence that ML can do better:**

**Multiple signals with complex interactions:**
- 44 columns of features (traffic, position, engagement, content depth, freshness, competition)
- Simple correlations with the label are weak (all < 0.4), suggesting no single dominant signal
- The best pages to prioritize likely need combinations: e.g., "high impressions + declining + good position + low CTR" or "thin content + visible + stale"

**Non-linear relationships:**
- Position 3→5 decline is more critical than position 15→17
- 1000→500 impression drop is more urgent than 50→25
- Log-transformations help, but a tree-based model can learn these thresholds automatically

**Reference pipeline proves ML works:**
- Hand-rule baseline achieves modest precision
- Random Forest substantially improves over hand-rule
- **ML learns interactions the fixed rule misses**

**But the reference pipeline left room for improvement:**
The current system uses:
- Basic feature set (no log transforms, no interaction terms)
- Standard Random Forest (no extensive hyperparameter tuning)
- Starter dataset only (no warehouse query-level features)
- Proxy label (current window comparison)

**My opportunities to improve:**
1. **Engineered features** — log(impressions), stale×visible, position×CTR, engagement depth ratios
2. **Advanced models** — XGBoost with tuned hyperparameters, ensemble methods
3. **Warehouse features** — query concentration, diversity, momentum from time-series
4. **Better label** — true forward-looking (month 1-3 → month 4) using warehouse data
5. **Feature selection** — remove noise, keep signal that generalizes across clients

The reference pipeline showed ML beats hand-rules. My goal is to show ML can do **even better** with improved features and models.

In [26]:
# Show evidence that the pattern is complex

# 1. Check correlations with label - are any single features dominant?
numeric_cols = df.select_dtypes(include=[np.number]).columns
# Exclude the label itself, IDs, and leakage columns
safe_cols = [c for c in numeric_cols if c not in ['trend_pct', 'content_id', 'client_id'] 
             and 'trend' not in c.lower()]

correlations = df[safe_cols].corrwith(df['is_declining_label']).abs().sort_values(ascending=False)

print("CORRELATION WITH LABEL (absolute value, top 10):")
print(correlations.head(10).to_string())
print()
print(f"Strongest single correlation: {correlations.iloc[0]:.3f}")
print("→ No single feature dominates; the pattern is distributed across many signals")
print()

# 2. Show interaction example: stale + visible pages
df['is_stale'] = df['days_since_last_update'] >= 180
df['is_visible'] = df['impressions_90d'] >= 500

print("INTERACTION EXAMPLE: Staleness + Visibility")
print(f"Stale pages (180+ days):        {df['is_stale'].sum():,} pages, {df[df['is_stale']]['is_declining_label'].mean()*100:.1f}% declining")
print(f"Visible pages (500+ impr):      {df['is_visible'].sum():,} pages, {df[df['is_visible']]['is_declining_label'].mean()*100:.1f}% declining")
print(f"Stale AND visible (interaction): {df[df['is_stale'] & df['is_visible']].shape[0]:,} pages, {df[df['is_stale'] & df['is_visible']]['is_declining_label'].mean()*100:.1f}% declining")
print()
print("→ The combination has different characteristics than either signal alone")
print()

# 3. Show non-linearity in position
position_bins = pd.cut(df[df['avg_position'] > 0]['avg_position'], bins=[0, 3, 10, 20, 50, 100], labels=['1-3', '4-10', '11-20', '21-50', '51-100'])
decline_by_position = df[df['avg_position'] > 0].groupby(position_bins)['is_declining_label'].mean()

print("NON-LINEAR RELATIONSHIP: Decline rate by position tier")
print(decline_by_position.to_string())
print()
print("→ Relationship is non-linear; a tree model can learn these thresholds")
print()

print("REFERENCE PIPELINE RESULTS:")
print("  Hand-rule baseline:  0.240 precision@50")
print("  Random Forest:       0.740 precision@50")
print("  Lift:                3.08× (ML learns interactions the rule misses)")

CORRELATION WITH LABEL (absolute value, top 10):
days_with_impressions     0.190055
content_age_days          0.163882
age_tier_order            0.156142
impressions_last_30d      0.093980
word_count                0.090157
days_since_last_update    0.081383
char_count                0.072188
clicks_last_30d           0.071935
demo_score                0.071624
sessions_last_30d         0.063842

Strongest single correlation: 0.190
→ No single feature dominates; the pattern is distributed across many signals

INTERACTION EXAMPLE: Staleness + Visibility
Stale pages (180+ days):        174 pages, 47.1% declining
Visible pages (500+ impr):      16,726 pages, 59.6% declining
Stale AND visible (interaction): 17 pages, 94.1% declining

→ The combination has different characteristics than either signal alone

NON-LINEAR RELATIONSHIP: Decline rate by position tier
avg_position
1-3       0.497809
4-10      0.569414
11-20     0.609515
21-50     0.561799
51-100    0.346420

→ Relationship is non-

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.